Processamento do arquivo

In [1]:
from collections import Counter
from pathlib import Path
import re

import pandas as pd

# Localiza o arquivo a partir da pasta do projeto.
csv_path = Path("sample/cases.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Arquivo nao encontrado: {csv_path.resolve()}")

# Carrega o CSV e aceita as duas formas mais comuns do nome da coluna.
df = pd.read_csv(csv_path)
text_column = next(
    (column for column in ("case_text", "case-text") if column in df.columns),
    None,
)
if text_column is None:
    raise KeyError(
        "A coluna de texto nao foi encontrada. "
        f"Colunas disponiveis: {list(df.columns)}"
    )

# Remove registros sem texto e cria uma versao tokenizada para analise.
df[text_column] = df[text_column].fillna("").astype(str)
df = df.loc[df[text_column].str.strip().ne("")].copy()
df["words"] = df[text_column].apply(
    lambda text: re.findall(r"[A-Za-z][A-Za-z'-]+", text.lower())
)

# Frequencia das palavras com pelo menos tres caracteres.
word_counts = Counter(
    word
    for words in df["words"]
    for word in words
    if len(word) >= 3
)
frequent_words = pd.DataFrame(
    word_counts.most_common(30),
    columns=["word", "frequency"],
)

print(f"Registros carregados: {len(df)}")
print(f"Coluna analisada: {text_column}")
display(df[[column for column in df.columns if column != "words"]].head())
display(frequent_words)

Registros carregados: 56
Coluna analisada: case_text


,article_id,age,case_id,case_text,gender
0,PMC5137649,44.0,PMC5137649_01,A 44-year-old woman presented with a 3-day his...,Female
1,PMC9387390,57.0,PMC9387390_01,A 57-year-old man with no significant past med...,Male
2,PMC3437073,55.0,PMC3437073_01,A 55-year-old male presented with a gradually ...,Male
3,PMC7102447,65.0,PMC7102447_01,A 65-year-old male with a past medical history...,Male
4,PMC6354154,30.0,PMC6354154_01,A 30-year-old nulligravida presented herself i...,Female


,word,frequency
0,the,1588
1,and,1020
2,was,765
3,with,496
4,patient,277
5,for,232
6,were,221
7,showed,128
8,she,122
9,had,114
